# **Retail Analytics**

<br></br>




In [108]:
# import required libraries
import duckdb as dd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

<br></br>


### 1. Creating Synthetic Data

In [109]:
# For reproducibility
np.random.seed(42)

In [110]:
# Create customer table with 500 rows
customer = pd.DataFrame({
"customer_id": range(1, 501),
"gender": np.random.choice(["Female","Male"],500),
"state" : np.random.choice(["California","Florida","New-York","Texas","Los Angeles"], 500),
"customer_age": np.random.choice(["18-24","25-34","35-44","45-54","55+"], 500),
"customer_segment": np.random.choice(["Budget", "Regular", "Premium"], 500)
})

customer.head()

,customer_id,gender,state,customer_age,customer_segment
0,1,Female,Los Angeles,18-24,Budget
1,2,Male,Los Angeles,55+,Regular
2,3,Female,California,45-54,Budget
3,4,Female,New-York,25-34,Budget
4,5,Female,Florida,18-24,Regular


In [111]:
# Create product table with 500 rows
product = pd.DataFrame({
"product_id": range(1, 501),
"unit_price": np.round(np.random.randint(5,500,500),2),
"brand": np.random.choice(["BrandA","BrandB","BrandC","BrandD","BrandE"], 500),
"category": np.random.choice(["Electronics","Clothing","Home","Sports","Books"], 500)
})

product.head()

,product_id,unit_price,brand,category
0,1,322,BrandC,Sports
1,2,320,BrandE,Electronics
2,3,385,BrandD,Books
3,4,115,BrandC,Electronics
4,5,18,BrandE,Electronics


In [112]:
# Create sales table with 10000 rows
sales = pd.DataFrame({
"transaction_id": range(1, 10001),
"customer_id": np.random.randint(1, 500,10000),
"product_id": np.random.randint(1,500,10000),
"quantity": np.random.randint(1,11,10000),
"sale_date": pd.to_datetime("2025-01-01")
+ pd.to_timedelta(np.random.randint(0,365,10000), unit="D")
})

sales.head()

,transaction_id,customer_id,product_id,quantity,sale_date
0,1,37,448,8,2025-12-08
1,2,170,264,5,2025-03-03
2,3,124,465,5,2025-06-23
3,4,357,223,10,2025-09-10
4,5,35,421,3,2025-04-22


In [113]:
# Join sales and product table
sales = pd.merge(sales,
    product[["product_id","unit_price"]],
    on = "product_id",
    how= "left"
)

# Now we can calculate a revenue column
sales["revenue"] = (sales["unit_price"] * sales["quantity"])

sales.head()

,transaction_id,customer_id,product_id,quantity,sale_date,unit_price,revenue
0,1,37,448,8,2025-12-08,134,1072
1,2,170,264,5,2025-03-03,431,2155
2,3,124,465,5,2025-06-23,330,1650
3,4,357,223,10,2025-09-10,295,2950
4,5,35,421,3,2025-04-22,306,918


In [114]:
# sales table dimension
sales.shape

(10000, 7)

<hr></hr>

### 2. SQL Queries Solution with DuckDB



In [129]:
# Count number of customers by state
dd.sql("""SELECT state, COUNT(*) AS customer_count
FROM customer
GROUP BY state
ORDER BY customer_count DESC;""").show()

┌─────────────┬────────────────┐
│    state    │ customer_count │
│   varchar   │     int64      │
├─────────────┼────────────────┤
│ Los Angeles │            114 │
│ Florida     │            110 │
│ California  │             99 │
│ Texas       │             91 │
│ New-York    │             86 │
└─────────────┴────────────────┘



In [133]:
#  Count the number of customers by customer segment
dd.sql("""SELECT customer_segment, COUNT(*) AS customer_count
FROM customer
GROUP BY customer_segment
ORDER BY customer_count DESC;""").show()

┌──────────────────┬────────────────┐
│ customer_segment │ customer_count │
│     varchar      │     int64      │
├──────────────────┼────────────────┤
│ Regular          │            174 │
│ Budget           │            171 │
│ Premium          │            155 │
└──────────────────┴────────────────┘



In [135]:
# Average unit price by product category
dd.sql("""SELECT ROUND(AVG(unit_price),2) AS avg_price, category
FROM product
GROUP BY category
ORDER BY avg_price DESC; """).show()

┌───────────┬─────────────┐
│ avg_price │  category   │
│  double   │   varchar   │
├───────────┼─────────────┤
│    271.18 │ Home        │
│    261.94 │ Clothing    │
│     257.5 │ Books       │
│    248.79 │ Sports      │
│    242.82 │ Electronics │
└───────────┴─────────────┘



In [138]:
# 10 most expensive products
dd.sql("""SELECT *
FROM product
ORDER BY unit_price DESC
LIMIT 10; """).show()

┌────────────┬────────────┬─────────┬──────────┐
│ product_id │ unit_price │  brand  │ category │
│   int64    │   int64    │ varchar │ varchar  │
├────────────┼────────────┼─────────┼──────────┤
│        108 │        499 │ BrandA  │ Books    │
│        133 │        499 │ BrandC  │ Sports   │
│        290 │        498 │ BrandE  │ Books    │
│        191 │        498 │ BrandB  │ Sports   │
│        430 │        498 │ BrandC  │ Clothing │
│        265 │        497 │ BrandB  │ Home     │
│         92 │        494 │ BrandD  │ Home     │
│        199 │        493 │ BrandA  │ Home     │
│        185 │        493 │ BrandA  │ Books    │
│        418 │        491 │ BrandC  │ Home     │
├────────────┴────────────┴─────────┴──────────┤
│ 10 rows                            4 columns │
└──────────────────────────────────────────────┘



In [141]:
# Total revenue generated
dd.sql("""SELECT SUM(revenue) AS total_revenue
FROM sales; """).show()

┌───────────────┐
│ total_revenue │
│    int128     │
├───────────────┤
│      14037058 │
└───────────────┘



In [144]:
# Total revenue by state
dd.sql("""SELECT SUM(revenue) AS total_revenue, state
FROM sales a
JOIN customer b
ON a.customer_id = b.customer_id
GROUP BY state
ORDER BY total_revenue DESC; """).show()

┌───────────────┬─────────────┐
│ total_revenue │    state    │
│    int128     │   varchar   │
├───────────────┼─────────────┤
│       3394541 │ Los Angeles │
│       3123062 │ Florida     │
│       2740626 │ California  │
│       2549947 │ Texas       │
│       2228882 │ New-York    │
└───────────────┴─────────────┘



In [145]:
# Total revenue by customer segment
dd.sql("""SELECT SUM(revenue) AS total_revenue, customer_segment
FROM sales a
JOIN customer b
ON a.customer_id = b.customer_id
GROUP BY customer_segment
ORDER BY total_revenue DESC; """).show()

┌───────────────┬──────────────────┐
│ total_revenue │ customer_segment │
│    int128     │     varchar      │
├───────────────┼──────────────────┤
│       4902796 │ Regular          │
│       4885212 │ Budget           │
│       4249050 │ Premium          │
└───────────────┴──────────────────┘



In [150]:
# Total revenue by product category
dd.sql("""SELECT SUM(revenue) AS total_revenue, category
FROM sales a
JOIN product b
ON a.product_id = b.product_id
GROUP BY category
ORDER BY total_revenue DESC; """).show()


┌───────────────┬─────────────┐
│ total_revenue │  category   │
│    int128     │   varchar   │
├───────────────┼─────────────┤
│       3104395 │ Home        │
│       3018509 │ Sports      │
│       2846988 │ Electronics │
│       2652591 │ Books       │
│       2414575 │ Clothing    │
└───────────────┴─────────────┘



In [151]:
# Total revenue by brand
dd.sql("""SELECT SUM(revenue) AS total_revenue, brand
FROM sales a
JOIN product b
ON a.product_id = b.product_id
GROUP BY brand
ORDER BY total_revenue DESC;""").show()


┌───────────────┬─────────┐
│ total_revenue │  brand  │
│    int128     │ varchar │
├───────────────┼─────────┤
│       3044063 │ BrandD  │
│       2940740 │ BrandE  │
│       2860228 │ BrandA  │
│       2843295 │ BrandB  │
│       2348732 │ BrandC  │
└───────────────┴─────────┘



In [167]:
# Find the top 10 customers by revenue
dd.sql("""SELECT SUM(a.revenue) AS total_revenue, a.customer_id
FROM sales a
JOIN customer b
ON a.customer_id = b.customer_id
GROUP BY a.customer_id
ORDER BY total_revenue DESC
LIMIT 10; """).show()

┌───────────────┬─────────────┐
│ total_revenue │ customer_id │
│    int128     │    int64    │
├───────────────┼─────────────┤
│         61522 │         337 │
│         58176 │         271 │
│         51160 │         480 │
│         50347 │         169 │
│         50177 │         313 │
│         48338 │         496 │
│         46878 │         273 │
│         46720 │         278 │
│         46487 │         331 │
│         45260 │         464 │
├───────────────┴─────────────┤
│ 10 rows           2 columns │
└─────────────────────────────┘



In [169]:
# Find the top 5 states by revenue
dd.sql("""SELECT SUM(a.revenue) AS total_revenue, state
FROM sales a
JOIN customer b
ON a.customer_id = b.customer_id
GROUP BY state
ORDER BY total_revenue DESC
LIMIT 5;""").show()


┌───────────────┬─────────────┐
│ total_revenue │    state    │
│    int128     │   varchar   │
├───────────────┼─────────────┤
│       3394541 │ Los Angeles │
│       3123062 │ Florida     │
│       2740626 │ California  │
│       2549947 │ Texas       │
│       2228882 │ New-York    │
└───────────────┴─────────────┘



In [178]:
# Calculate average order value (AOV).
#THis metric measures how much money the average customer spends each time they place an order
dd.sql("""SELECT ROUND(SUM(revenue)/ COUNT(DISTINCT transaction_id),2) AS avg_order_value
FROM sales; """).show()

┌─────────────────┐
│ avg_order_value │
│     double      │
├─────────────────┤
│         1403.71 │
└─────────────────┘



In [197]:
# Find average monthly revenue trend
dd.sql("""SELECT ROUND(AVG(revenue),2) AS avg_revenue, CAST(sale_date AS DATE) AS sale_date_only
FROM sales
GROUP BY sale_date_only
ORDER BY avg_revenue DESC
LIMIT 20;""").show()

┌─────────────┬────────────────┐
│ avg_revenue │ sale_date_only │
│   double    │      date      │
├─────────────┼────────────────┤
│     1955.55 │ 2025-06-25     │
│     1942.54 │ 2025-11-19     │
│     1924.88 │ 2025-04-20     │
│      1918.9 │ 2025-04-11     │
│     1914.71 │ 2025-12-10     │
│     1893.82 │ 2025-06-16     │
│     1877.35 │ 2025-05-15     │
│     1866.74 │ 2025-03-03     │
│     1866.54 │ 2025-10-22     │
│     1821.04 │ 2025-09-24     │
│     1805.19 │ 2025-11-27     │
│     1797.71 │ 2025-07-02     │
│     1794.38 │ 2025-12-12     │
│     1791.54 │ 2025-11-26     │
│      1784.1 │ 2025-08-25     │
│     1778.17 │ 2025-02-11     │
│     1773.17 │ 2025-05-25     │
│     1767.36 │ 2025-12-29     │
│     1762.21 │ 2025-02-21     │
│     1761.87 │ 2025-03-11     │
├─────────────┴────────────────┤
│ 20 rows            2 columns │
└──────────────────────────────┘



In [198]:
# Find average quarterly revenue trend
dd.sql("""SELECT ROUND(AVG(revenue),2) AS avg_revenue, DATE_TRUNC('quarter', sale_date) AS quarter
FROM sales
GROUP BY quarter
ORDER BY avg_revenue DESC
LIMIT 20;""").show()

┌─────────────┬────────────┐
│ avg_revenue │  quarter   │
│   double    │    date    │
├─────────────┼────────────┤
│     1425.16 │ 2025-10-01 │
│     1418.15 │ 2025-04-01 │
│     1401.28 │ 2025-07-01 │
│      1370.4 │ 2025-01-01 │
└─────────────┴────────────┘



In [242]:
# Customer who never purchased electronic products
dd.sql("""
    SELECT
        c.customer_id,
        c.state
    FROM customer c
    WHERE c.customer_id NOT IN (
        SELECT DISTINCT s.customer_id
        FROM sales s
        JOIN product p
            ON s.product_id = p.product_id
        WHERE p.category = 'Electronics'
    )
    ORDER BY c.customer_id;
""").show()


┌─────────────┬────────────┐
│ customer_id │   state    │
│    int64    │  varchar   │
├─────────────┼────────────┤
│          33 │ California │
│          60 │ New-York   │
│         142 │ Florida    │
│         265 │ New-York   │
│         284 │ California │
│         429 │ California │
│         438 │ New-York   │
│         457 │ Florida    │
│         475 │ California │
│         500 │ New-York   │
├─────────────┴────────────┤
│ 10 rows        2 columns │
└──────────────────────────┘



In [201]:
# Which age group generates the most revenue?
dd.sql("""SELECT ROUND(AVG(a.revenue),2) AS avg_revenue, customer_age
FROM sales a
JOIN customer b
ON a.customer_id = b.customer_id
GROUP BY b.customer_age
ORDER BY avg_revenue DESC;  """).show()


┌─────────────┬──────────────┐
│ avg_revenue │ customer_age │
│   double    │   varchar    │
├─────────────┼──────────────┤
│     1414.71 │ 35-44        │
│     1408.79 │ 18-24        │
│     1401.71 │ 25-34        │
│     1400.35 │ 55+          │
│     1389.89 │ 45-54        │
└─────────────┴──────────────┘



In [202]:
# Which gender generates the most revenue?
dd.sql("""SELECT ROUND(AVG(a.revenue),2) AS avg_revenue, gender
FROM sales a
JOIN customer b
ON a.customer_id = b.customer_id
GROUP BY gender
ORDER BY avg_revenue DESC;  """).show()

┌─────────────┬─────────┐
│ avg_revenue │ gender  │
│   double    │ varchar │
├─────────────┼─────────┤
│     1411.22 │ Female  │
│     1396.64 │ Male    │
└─────────────┴─────────┘



In [236]:
#Find the percentage contribution of each category to total revenue
dd.sql("""
    SELECT
        p.category,
        SUM(s.revenue) AS total_revenue,
        ROUND(SUM(s.revenue) * 100.0
            / SUM(SUM(s.revenue)) OVER (),2) AS revenue_pct
    FROM sales s
    JOIN product p
        ON s.product_id = p.product_id
    GROUP BY p.category
    ORDER BY revenue_pct DESC;
""").show()


┌─────────────┬───────────────┬─────────────┐
│  category   │ total_revenue │ revenue_pct │
│   varchar   │    int128     │   double    │
├─────────────┼───────────────┼─────────────┤
│ Home        │       3104395 │       22.12 │
│ Sports      │       3018509 │        21.5 │
│ Electronics │       2846988 │       20.28 │
│ Books       │       2652591 │        18.9 │
│ Clothing    │       2414575 │        17.2 │
└─────────────┴───────────────┴─────────────┘



In [206]:
# Highest revenue category within each state
dd.sql(""" WITH category_state_revenue AS(
SELECT SUM(a.revenue) AS total_revenue, b.category, c.state
FROM sales a
JOIN customer c
ON a.customer_id = c.customer_id
JOIN product b
ON a.product_id = b.product_id
GROUP BY c.state, b.category)
SELECT * FROM category_state_revenue
QUALIFY ROW_NUMBER() OVER(PARTITION BY state ORDER BY total_revenue DESC)=1
ORDER BY state;  """).show()

┌───────────────┬─────────────┬─────────────┐
│ total_revenue │  category   │    state    │
│    int128     │   varchar   │   varchar   │
├───────────────┼─────────────┼─────────────┤
│        664764 │ Sports      │ California  │
│        727526 │ Sports      │ Florida     │
│        801418 │ Home        │ Los Angeles │
│        473017 │ Electronics │ New-York    │
│        576238 │ Home        │ Texas       │
└───────────────┴─────────────┴─────────────┘



In [209]:
# Rank products by revenue within each category
dd.sql("""WITH product_category_revenue AS(
SELECT SUM(a.revenue) AS total_revenue, b.category, a.product_id
FROM  sales a
JOIN product b
ON a.product_id = b.product_id
GROUP BY b.category, a.product_id)
SELECT category, product_id, total_revenue,
RANK() OVER(PARTITION BY category
ORDER BY total_revenue DESC) AS revenue_rank
FROM product_category_revenue
ORDER BY category, revenue_rank;""").show()

┌──────────┬────────────┬───────────────┬──────────────┐
│ category │ product_id │ total_revenue │ revenue_rank │
│ varchar  │   int64    │    int128     │    int64     │
├──────────┼────────────┼───────────────┼──────────────┤
│ Books    │        290 │         74700 │            1 │
│ Books    │         13 │         67791 │            2 │
│ Books    │         51 │         63720 │            3 │
│ Books    │        308 │         62658 │            4 │
│ Books    │        393 │         61920 │            5 │
│ Books    │         26 │         60417 │            6 │
│ Books    │        318 │         57728 │            7 │
│ Books    │        185 │         57681 │            8 │
│ Books    │        111 │         57540 │            9 │
│ Books    │        108 │         55389 │           10 │
│   ·      │          · │            ·  │            · │
│   ·      │          · │            ·  │            · │
│   ·      │          · │            ·  │            · │
│ Sports   │         49 │      

In [237]:
# Compare revenue between Premium and Budget customers
dd.sql("""
    SELECT
        c.customer_segment,
        SUM(a.revenue) AS total_revenue,
        COUNT(DISTINCT a.transaction_id) AS total_orders,
        ROUND(SUM(a.revenue) * 1.0 / COUNT(DISTINCT a.transaction_id),2) AS avg_order_value
    FROM sales a
    JOIN customer c
        ON a.customer_id = c.customer_id
    WHERE c.customer_segment IN ('Premium', 'Budget')
    GROUP BY c.customer_segment
    ORDER BY total_revenue DESC;
""").show()


┌──────────────────┬───────────────┬──────────────┬─────────────────┐
│ customer_segment │ total_revenue │ total_orders │ avg_order_value │
│     varchar      │    int128     │    int64     │     double      │
├──────────────────┼───────────────┼──────────────┼─────────────────┤
│ Budget           │       4885212 │         3356 │         1455.67 │
│ Premium          │       4249050 │         3115 │         1364.06 │
└──────────────────┴───────────────┴──────────────┴─────────────────┘



In [217]:
# Calculate cumulative revenue over time
dd.sql("""
    WITH monthly AS (
        SELECT
            DATE_TRUNC('month', sale_date) AS month,
            SUM(revenue) AS monthly_revenue
        FROM sales
        GROUP BY month
    )
    SELECT
        month,
        monthly_revenue,
        SUM(monthly_revenue) OVER (
            ORDER BY month
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_revenue
    FROM monthly
    ORDER BY month;
""").show()


┌────────────┬───────────────┬────────────────────┐
│ date_only  │ daily_revenue │ cumulative_revenue │
│    date    │    int128     │       int128       │
├────────────┼───────────────┼────────────────────┤
│ 2025-01-01 │       1176492 │            1176492 │
│ 2025-02-01 │       1067806 │            2244298 │
│ 2025-03-01 │       1196766 │            3441064 │
│ 2025-04-01 │       1217583 │            4658647 │
│ 2025-05-01 │       1135822 │            5794469 │
│ 2025-06-01 │       1177796 │            6972265 │
│ 2025-07-01 │       1170451 │            8142716 │
│ 2025-08-01 │       1185100 │            9327816 │
│ 2025-09-01 │       1142055 │           10469871 │
│ 2025-10-01 │       1115028 │           11584899 │
│ 2025-11-01 │       1212469 │           12797368 │
│ 2025-12-01 │       1239690 │           14037058 │
├────────────┴───────────────┴────────────────────┤
│ 12 rows                               3 columns │
└─────────────────────────────────────────────────┘



In [239]:
#----------------------------- RFM-style customer analysis---------------------------
#-- Recency: days since last purchase
#-- Frequency: How many orders customers have placed
#-- Monetary: Total revenue generated

dd.sql("""
    WITH base AS (
        SELECT
            customer_id,
            CAST(MAX(sale_date) AS DATE) AS last_purchase,
            COUNT(DISTINCT transaction_id) AS frequency,
            SUM(revenue) AS monetary
        FROM sales
        GROUP BY customer_id
    ),
    rfm AS (
        SELECT
            customer_id,
            DATE_DIFF('day', last_purchase, CURRENT_DATE) AS recency,
            frequency,
            monetary
        FROM base
    )
    SELECT *
    FROM rfm
    ORDER BY monetary DESC;
""").show()


┌─────────────┬─────────┬───────────┬──────────┐
│ customer_id │ recency │ frequency │ monetary │
│    int64    │  int64  │   int64   │  int128  │
├─────────────┼─────────┼───────────┼──────────┤
│         337 │     178 │        36 │    61522 │
│         271 │     191 │        37 │    58176 │
│         480 │     173 │        32 │    51160 │
│         169 │     183 │        23 │    50347 │
│         313 │     181 │        35 │    50177 │
│         496 │     173 │        23 │    48338 │
│         273 │     179 │        34 │    46878 │
│         278 │     174 │        29 │    46720 │
│         331 │     179 │        25 │    46487 │
│         464 │     197 │        25 │    45260 │
│          ·  │      ·  │         · │      ·   │
│          ·  │      ·  │         · │      ·   │
│          ·  │      ·  │         · │      ·   │
│         243 │     179 │        10 │    11825 │
│         454 │     224 │        10 │    11672 │
│         142 │     172 │         8 │    11141 │
│          60 │     

In [241]:
# Create a view for monthly sales summaries
dd.sql("""
    CREATE OR REPLACE VIEW monthly_sales_summary AS
    SELECT
        DATE_TRUNC('month', sale_date) AS month,
        SUM(revenue) AS total_revenue_generated,
        SUM(quantity) AS total_quantity_sold,
        COUNT(DISTINCT transaction_id) AS total_orders,
        ROUND(SUM(revenue) * 1.0 / COUNT(DISTINCT transaction_id),2) AS avg_order_value
    FROM sales
    GROUP BY month
    ORDER BY month;
""")

# Query the created summary
dd.sql("""SELECT * FROM monthly_sales_summary; """).show()

┌────────────┬─────────────────────────┬─────────────────────┬──────────────┬─────────────────┐
│   month    │ total_revenue_generated │ total_quantity_sold │ total_orders │ avg_order_value │
│    date    │         int128          │       int128        │    int64     │     double      │
├────────────┼─────────────────────────┼─────────────────────┼──────────────┼─────────────────┤
│ 2025-01-01 │                 1176492 │                4769 │          897 │         1311.59 │
│ 2025-02-01 │                 1067806 │                4229 │          769 │         1388.56 │
│ 2025-03-01 │                 1196766 │                4657 │          845 │         1416.29 │
│ 2025-04-01 │                 1217583 │                4714 │          850 │         1432.45 │
│ 2025-05-01 │                 1135822 │                4416 │          825 │         1376.75 │
│ 2025-06-01 │                 1177796 │                4486 │          815 │         1445.15 │
│ 2025-07-01 │                 1170451 │